In [ ]:
# Environment setup (high-compute friendly)
# - Installs runtime dependencies
# - Provides a robust shell runner for diagnostics

import os
import shlex
import subprocess
import sys
from pathlib import Path


def run_cmd(cmd: list[str], *, cwd: str | Path | None = None) -> str:
    """Run a command and return combined stdout/stderr; raise on non-zero exit."""
    display = " ".join(shlex.quote(str(c)) for c in cmd)
    print(f"\n$ {display}")
    p = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    out = p.stdout or ""
    if out.strip():
        print(out)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed (exit_code={p.returncode}): {display}")
    return out


os.environ.setdefault("PYTHONWARNINGS", "default")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

run_cmd([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "yfinance",
        "pyarrow",
        "torch",
        "scikit-learn",
        "tqdm",
        "hmmlearn",
        "joblib",
        "matplotlib",
        "scipy",
    ]
)

print("Python:", sys.version.split()[0])
print("CWD:", Path.cwd().resolve())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
# GPU diagnostics + PyTorch device configuration

import platform

print("Platform:", platform.platform())

try:
    _ = run_cmd(["bash", "-lc", "nvidia-smi"])  # common on GPU images
except Exception as e:
    print("nvidia-smi unavailable:", repr(e))

import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device 0:", torch.cuda.get_device_name(0))

# Determinism knobs (best-effort; trades a bit of speed for repeatability)
import random
import numpy as np

def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy, and PyTorch RNGs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# =========================
# RAMT Monolith — Library
# =========================
# This section consolidates the project into notebook-local, import-free building blocks.

from __future__ import annotations

import contextlib
import io
import math
import time
import warnings
import zipfile
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import RobustScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from scipy.stats import rankdata


# -------------------------
# Data ingestion (NIFTY 200)
# -------------------------

NSE_NIFTY200_CSV_URL = "https://archives.nseindia.com/content/indices/ind_nifty200list.csv"

RAW_REQUIRED_COLS = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

MACRO_TICKERS: dict[str, str] = {
    "INDIAVIX": "^INDIAVIX",
    "CRUDE": "CL=F",
    "USDINR": "INR=X",
    "SP500": "^GSPC",
}


def _flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)
    out.columns = [str(c).strip() for c in out.columns]
    return out


def _safe_stem(text: str) -> str:
    return (
        str(text)
        .replace(".", "_")
        .replace("^", "_")
        .replace("=", "_")
        .replace("/", "_")
        .replace("&", "_")
        .replace("-", "_")
        .strip("_")
    )


def fetch_nifty200_symbols() -> list[str]:
    """Fetch the current NIFTY 200 constituent snapshot and return Yahoo tickers like `RELIANCE.NS`."""
    warnings.warn(
        "Fetching the current NIFTY 200 constituent snapshot. "
        "Historical index membership is not reconstructed (survivorship bias may remain).",
        RuntimeWarning,
        stacklevel=2,
    )
    table = pd.read_csv(NSE_NIFTY200_CSV_URL)
    if "Symbol" not in table.columns:
        raise RuntimeError(f"NSE CSV missing Symbol column. Columns: {list(table.columns)}")

    symbols = (
        table["Symbol"]
        .astype(str)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA})
        .dropna()
        .unique()
        .tolist()
    )
    symbols = sorted(symbols, key=str)
    return [f"{s}.NS" if not str(s).endswith(".NS") else str(s) for s in symbols]


def _download_daily_ohlcv_adjclose(ticker: str, start: str, end_exclusive: str) -> pd.DataFrame:
    """Download daily OHLCV + Adj Close from yfinance with a small retry loop."""
    raw = pd.DataFrame()
    last_err: Exception | None = None
    for attempt in range(3):
        try:
            raw = yf.download(
                ticker,
                start=start,
                end=end_exclusive,
                interval="1d",
                auto_adjust=False,
                actions=False,
                threads=False,
                progress=False,
            )
            if not raw.empty:
                break
            raw = yf.Ticker(ticker).history(
                start=start,
                end=end_exclusive,
                interval="1d",
                auto_adjust=False,
                actions=False,
            )
            if not raw.empty:
                break
        except Exception as e:
            last_err = e
        time.sleep(1.0 + attempt * 2.0)

    if raw.empty:
        if last_err is not None:
            raise RuntimeError(f"yfinance failed for {ticker}: {last_err}") from last_err
        raise RuntimeError(f"yfinance returned empty data for {ticker}")

    df = _flatten_yfinance_columns(raw)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise RuntimeError(f"{ticker}: missing required columns: {missing}")

    df = df[REQUIRED_COLS].copy().reset_index()
    if "Date" not in df.columns and "Datetime" in df.columns:
        df = df.rename(columns={"Datetime": "Date"})
    if "Date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Date"})

    df["Date"] = pd.to_datetime(df["Date"], utc=False).dt.tz_localize(None)
    df.insert(0, "Ticker", ticker)
    return df


def download_universe_to_parquet(
    *,
    raw_dir: str | Path,
    start: str,
    end_exclusive: str,
    rate_limit_s: float = 0.5,
) -> dict[str, object]:
    """Download NIFTY200 equities + macro series into `raw_dir` as Parquet (one file per series)."""
    raw_root = Path(raw_dir)
    raw_root.mkdir(parents=True, exist_ok=True)

    tickers = fetch_nifty200_symbols()
    series: list[tuple[str, str]] = [(t, t) for t in tickers]
    series.extend([(f"macro_{k}", v) for k, v in MACRO_TICKERS.items()])

    ok = 0
    failed: list[tuple[str, str]] = []

    pbar = tqdm(series, desc="download", mininterval=0.5)
    for name, yf_ticker in pbar:
        try:
            df = _download_daily_ohlcv_adjclose(yf_ticker, start, end_exclusive)
            out_path = raw_root / f"{_safe_stem(yf_ticker) if name == yf_ticker else _safe_stem(name)}.parquet"
            df.to_parquet(out_path, index=False)
            ok += 1
            pbar.set_postfix(ok=ok, last=str(out_path.name), refresh=False)
        except Exception as e:
            failed.append((yf_ticker, str(e)))
            pbar.set_postfix(fail=len(failed), last=name, refresh=False)
        time.sleep(float(rate_limit_s))

    return {"raw_dir": str(raw_root.resolve()), "tickers": tickers, "ok": ok, "failed": failed}


# -------------------------
# Feature engineering (causal)
# -------------------------

HMM_MIN_OBS = 60
FEATURE_OUTPUT_COLUMNS: list[str] = [
    "Date",
    "Ticker",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
    "Ret_1d",
    "Ret_5d",
    "Ret_21d",
    "Realized_Vol_20",
    "RSI_14",
    "BB_Dist",
    "Volume_Surge",
] + [f"Macro_{k}_Ret1d_L1" for k in MACRO_TICKERS.keys()] + [
    "Monthly_Alpha",
    "Daily_Return",
    "HMM_Regime",
]


def _read_raw_parquet(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    missing = [c for c in RAW_REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; have {list(df.columns)}")
    df = df.sort_values("Date").reset_index(drop=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    return df


def _read_raw_equity_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    df = df.sort_values("Date").reset_index(drop=True)
    if "Adj Close" not in df.columns and "Close" in df.columns:
        df["Adj Close"] = df["Close"].astype(float)
    for col in ("Open", "High", "Low", "Close"):
        if col not in df.columns and "Adj Close" in df.columns:
            df[col] = df["Adj Close"]
    if "Volume" not in df.columns:
        df["Volume"] = 1.0
    if "Ticker" not in df.columns:
        df.insert(0, "Ticker", "^NSEI")
    missing = [c for c in RAW_REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; have {list(df.columns)}")
    return df


def _read_raw_equity(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return _read_raw_equity_csv(path)
    if suf == ".parquet":
        return _read_raw_parquet(path)
    raise ValueError(f"Unsupported raw equity path: {path}")


def list_stock_parquet_files(raw_dir: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(raw_dir.glob("*.parquet")):
        if p.name.startswith("macro_"):
            continue
        out.append(p)
    return out


def list_equity_input_paths(raw_dir: Path) -> list[Path]:
    paths = list_stock_parquet_files(raw_dir)
    nse_pq = raw_dir / "_NSEI.parquet"
    nse_csv = raw_dir / "_NSEI_raw.csv"
    if not nse_pq.exists() and nse_csv.exists():
        paths = [p for p in paths if p.name != "_NSEI_raw.csv"]
        paths.append(nse_csv)
    return sorted(set(paths))


def load_macro_series(raw_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Load macro Parquet files under `raw_dir` into a dict keyed by canonical `MACRO_TICKERS` names."""
    raw_root = Path(raw_dir)
    macro_paths = list(sorted(raw_root.glob("macro_*.parquet")))
    if not macro_paths:
        raise FileNotFoundError(f"No macro parquet files found in: {raw_root.resolve()}")

    out: dict[str, pd.DataFrame] = {}
    for p in macro_paths:
        df = _read_raw_parquet(p)
        pu = p.name.upper()
        matched = None
        for nm in MACRO_TICKERS.keys():
            if nm in pu:
                matched = nm
                break
        if matched is None:
            matched = p.name.split("macro_", 1)[1].split("_", 1)[0].upper()
        out[matched] = df.set_index("Date", drop=False)

    missing = [k for k in MACRO_TICKERS.keys() if k not in out]
    if missing:
        raise FileNotFoundError(
            f"Missing required macro series {missing}. Found: {sorted(out.keys())}. "
            f"Expected files for {sorted(MACRO_TICKERS.keys())} in {raw_root.resolve()}."
        )
    return out


def add_returns_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    px = out["Adj Close"].astype(float).replace(0.0, np.nan)
    r1 = px / px.shift(1)
    r5 = px / px.shift(5)
    r21 = px / px.shift(21)
    out["Ret_1d"] = np.log(r1.where(r1 > 0.0))
    out["Ret_5d"] = np.log(r5.where(r5 > 0.0))
    out["Ret_21d"] = np.log(r21.where(r21 > 0.0))
    return out


def add_realized_vol_20(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Realized_Vol_20"] = out["Ret_1d"].rolling(20).std()
    return out


def add_daily_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Daily_Return"] = out["Ret_1d"].shift(-1)
    return out


def add_rsi_14(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    close = out["Adj Close"].astype(float)
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    alpha = 1.0 / 14.0
    avg_gain = gain.ewm(alpha=alpha, adjust=False).mean()
    avg_loss = loss.ewm(alpha=alpha, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.where(avg_loss != 0.0, 100.0)
    out["RSI_14"] = rsi
    return out


def add_bollinger_distance(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    px = out["Adj Close"].astype(float)
    ma20 = px.rolling(20).mean()
    std20 = px.rolling(20).std()
    denom = (2.0 * std20).replace(0.0, np.nan)
    out["BB_Dist"] = (px - ma20) / denom
    return out


def add_volume_surge(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    vol = out["Volume"].astype(float)
    sma20 = vol.rolling(20).mean().replace(0.0, np.nan)
    out["Volume_Surge"] = vol / sma20
    return out


def add_macro_lagged_returns(df: pd.DataFrame, macro: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Merge macro 1d log returns with 1-day lag to avoid leakage (features/feature_engineering.add_macro_lagged_returns)."""
    out = df.copy()
    out = out.set_index("Date", drop=False)
    for name in MACRO_TICKERS.keys():
        mdf = macro[name]
        px = mdf["Adj Close"].astype(float).replace(0.0, np.nan)
        r1 = px / px.shift(1)
        mret = np.log(r1.where(r1 > 0.0)).shift(1)
        mret = mret.rename(f"Macro_{name}_Ret1d_L1")
        out = out.join(mret, how="left")
        out[mret.name] = out[mret.name].ffill().fillna(0.0)
    return out.reset_index(drop=True)


def compute_monthly_alpha_adjclose(df: pd.DataFrame, nifty: pd.DataFrame) -> pd.DataFrame:
    """Monthly_Alpha = ln(P_{t+21}/P_t) - ln(N_{t+21}/N_t) (features/feature_engineering.compute_monthly_alpha_adjclose)."""
    out = df.copy()
    out = out.set_index("Date", drop=False)
    nifty = nifty.set_index("Date", drop=False)

    p = out["Adj Close"].astype(float).replace(0.0, np.nan)
    r_fwd = p.shift(-21) / p
    stock_fwd = np.log(r_fwd.where(r_fwd > 0.0))

    aligned = out.join(nifty[["Adj Close"]].rename(columns={"Adj Close": "NIFTY_AdjClose"}), how="left")
    n = aligned["NIFTY_AdjClose"].astype(float).replace(0.0, np.nan)
    n_fwd = n.shift(-21) / n
    nifty_fwd = np.log(n_fwd.where(n_fwd > 0.0))

    out["Monthly_Alpha"] = stock_fwd - nifty_fwd
    return out.reset_index(drop=True)


def _semantic_hmm_mapping(mean_by_state: dict[int, float]) -> dict[int, int]:
    active = {int(k): float(v) for k, v in mean_by_state.items() if np.isfinite(v)}
    if not active:
        raise ValueError("Cannot map semantic HMM states without any active states.")

    states = sorted(active.keys(), key=lambda s: active[s])
    if len(states) == 1:
        return {states[0]: 0}
    if len(states) == 2:
        low, high = states[0], states[1]
        return {high: 1, low: 2}

    low = states[0]
    high = states[-1]
    middle_states = states[1:-1]
    mid = middle_states[len(middle_states) // 2]

    mapping = {high: 1, low: 2}
    for s in middle_states:
        mapping[s] = 0
    mapping[mid] = 0
    return mapping


def _build_gaussian_hmm(prev_hmm: GaussianHMM | None = None) -> GaussianHMM:
    init_params = "stmc" if prev_hmm is None else ""
    hmm = GaussianHMM(
        n_components=3,
        covariance_type="diag",
        n_iter=300,
        tol=1e-4,
        random_state=42,
        verbose=False,
        init_params=init_params,
        params="stmc",
    )
    if prev_hmm is not None:
        hmm.startprob_ = prev_hmm.startprob_.copy()
        hmm.transmat_ = prev_hmm.transmat_.copy()
        hmm.means_ = prev_hmm.means_.copy()
        hmm._covars_ = prev_hmm._covars_.copy()
    return hmm


def _fit_hmm_silently(hmm: GaussianHMM, X: np.ndarray) -> GaussianHMM:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        hmm.fit(X)
    return hmm


def add_hmm_regime_full_history(df: pd.DataFrame) -> pd.DataFrame:
    """Causal expanding-window 3-state Gaussian HMM on [Ret_1d, Realized_Vol_20]."""
    out = df.copy()
    lr = out["Ret_1d"]
    rv20 = out["Realized_Vol_20"]
    mask = lr.notna() & rv20.notna()

    # HMM fallback: default to Neutral (0) so we never drop a stock due to HMM issues.
    out["HMM_Regime"] = 0.0

    if int(mask.sum()) < HMM_MIN_OBS:
        return out

    idx = out.index[mask]
    lr_vals = lr[mask].to_numpy(dtype=float)
    rv_vals = rv20[mask].to_numpy(dtype=float)
    X_raw = np.column_stack([lr_vals, rv_vals])

    prev_hmm: GaussianHMM | None = None
    prev_regime = float("nan")
    fit_failures = 0

    for end_pos in range(HMM_MIN_OBS - 1, len(idx)):
        X_hist_raw = X_raw[: end_pos + 1]
        mu = X_hist_raw.mean(axis=0)
        sigma = X_hist_raw.std(axis=0)
        sigma = np.where(sigma == 0.0, 1.0, sigma)
        X_hist = (X_hist_raw - mu) / sigma

        try:
            hmm = _build_gaussian_hmm(prev_hmm)
            hmm = _fit_hmm_silently(hmm, X_hist)
            raw_states = hmm.predict(X_hist)

            means: dict[int, float] = {}
            lr_hist = lr_vals[: end_pos + 1]
            for k in (0, 1, 2):
                sel = raw_states == k
                means[k] = float(np.mean(lr_hist[sel])) if np.any(sel) else float("nan")

            mapping = _semantic_hmm_mapping(means)
            prev_regime = float(mapping[int(raw_states[-1])])
            prev_hmm = hmm
        except Exception:
            fit_failures += 1
            if not np.isfinite(prev_regime):
                prev_regime = 0.0

        out.loc[idx[end_pos], "HMM_Regime"] = prev_regime

    if fit_failures:
        warnings.warn(
            f"Causal HMM fit fell back to the prior regime {fit_failures} time(s).",
            RuntimeWarning,
            stacklevel=2,
        )

    out["HMM_Regime"] = out["HMM_Regime"].astype("float")
    return out


def build_features_table(
    df: pd.DataFrame, nifty_df: pd.DataFrame, macro_data: dict[str, pd.DataFrame]
) -> pd.DataFrame:
    out = df.copy()
    out = add_returns_features(out)
    out = add_realized_vol_20(out)
    out = add_rsi_14(out)
    out = add_bollinger_distance(out)
    out = add_volume_surge(out)
    out = add_macro_lagged_returns(out, macro_data)
    out = compute_monthly_alpha_adjclose(out, nifty_df)
    out = add_daily_target(out)
    out = add_hmm_regime_full_history(out)

    # Warm-up NaNs are expected; keep rows and use neutral defaults.
    fill_zero_cols = [
        "Ret_1d",
        "Ret_5d",
        "Ret_21d",
        "Realized_Vol_20",
        "RSI_14",
        "BB_Dist",
        "Volume_Surge",
        "Daily_Return",
        "HMM_Regime",
    ] + [f"Macro_{k}_Ret1d_L1" for k in MACRO_TICKERS.keys()]
    for c in fill_zero_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out[FEATURE_OUTPUT_COLUMNS]


def _calendar_from_benchmark(nifty_df: pd.DataFrame, *, start: str, end_exclusive: str) -> pd.DatetimeIndex:
    """Trading calendar from benchmark; matches features/feature_engineering._calendar_from_benchmark (half-open end)."""
    cal = pd.to_datetime(nifty_df["Date"]).dt.tz_localize(None)
    cal = cal[(cal >= pd.Timestamp(start)) & (cal < pd.Timestamp(end_exclusive))]
    return pd.DatetimeIndex(cal.unique()).sort_values()


def _align_equity_to_calendar(df: pd.DataFrame, calendar: pd.DatetimeIndex, ticker: str) -> pd.DataFrame:
    """Align equity OHLCV to benchmark calendar; matches features/feature_engineering._align_equity_to_calendar."""
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"]).dt.tz_localize(None)
    out = out.sort_values("Date").drop_duplicates(subset=["Date"]).set_index("Date", drop=False)

    out = out.reindex(calendar)
    out["Date"] = out.index
    out["Ticker"] = ticker

    price_cols = ["Open", "High", "Low", "Close", "Adj Close"]
    for c in price_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["Volume"] = pd.to_numeric(out["Volume"], errors="coerce")

    out[price_cols] = out[price_cols].ffill()

    if out["Adj Close"].isna().any():
        first_valid = out["Adj Close"].first_valid_index()
        if first_valid is not None:
            first_row = out.loc[first_valid, price_cols]
            out.loc[:first_valid, price_cols] = out.loc[:first_valid, price_cols].fillna(first_row)

    out["Volume"] = out["Volume"].fillna(0.0)
    return out.reset_index(drop=True)


def process_features_to_parquet(
    *,
    raw_dir: str | Path,
    processed_dir: str | Path,
    start: str,
    end_inclusive: str,
) -> dict[str, object]:
    """Transform raw Parquet series into processed feature Parquet files."""
    raw_root = Path(raw_dir)
    proc_root = Path(processed_dir)
    proc_root.mkdir(parents=True, exist_ok=True)

    end_exclusive = (pd.Timestamp(end_inclusive) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    # Load benchmark (^NSEI) from raw_dir; download if missing.
    nifty_path = raw_root / "_NSEI.parquet"
    if not nifty_path.exists():
        df = _download_daily_ohlcv_adjclose(
            "^NSEI",
            start,
            end_exclusive,
        )
        df.to_parquet(nifty_path, index=False)

    nifty_df = _read_raw_equity(nifty_path)
    macro = load_macro_series(raw_root)

    cal = _calendar_from_benchmark(nifty_df, start=start, end_exclusive=end_exclusive)

    equity_paths = list_equity_input_paths(raw_root)
    written = 0
    skipped = 0

    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end_inclusive)

    pbar = tqdm(equity_paths, desc="features", mininterval=0.5)
    for p in pbar:
        df = _read_raw_equity(p)
        df = df[(df["Date"] >= start_ts) & (df["Date"] <= end_ts)]
        if df.empty:
            skipped += 1
            print(f"Skipping {p.stem}: no rows in [{start}..{end_inclusive}]")
            continue

        ticker = str(df["Ticker"].iloc[0]) if "Ticker" in df.columns else p.stem

        # Loosen date filter: align to calendar and pad beginning if the stock starts late.
        df = _align_equity_to_calendar(df, cal, ticker)

        out_raw = build_features_table(df, nifty_df, macro)
        out = out_raw.dropna(subset=["Monthly_Alpha"])  # only require training label

        if out.empty:
            skipped += 1
            nan_fracs = out_raw.isna().mean(numeric_only=False).sort_values(ascending=False)
            top = ", ".join(f"{k}={v*100:.1f}%" for k, v in nan_fracs.head(5).items())
            rsi_nan = float(out_raw["RSI_14"].isna().mean()) * 100.0 if "RSI_14" in out_raw.columns else 0.0
            print(f"Skipping {ticker}: 0 rows with Monthly_Alpha. RSI NaNs={rsi_nan:.1f}%. Top NaNs: {top}")
            continue

        out_path = proc_root / f"{_safe_stem(ticker)}_features.parquet"
        out.to_parquet(out_path, index=False)
        written += 1
        pbar.set_postfix(written=written, refresh=False)

    return {"processed_dir": str(proc_root.resolve()), "written": written, "skipped": skipped}


# -------------------------
# Dataset engine (no bfill)
# -------------------------

PRICE_COLS = ["Ret_1d", "Ret_5d", "Ret_21d"]
TECH_COLS = ["RSI_14", "BB_Dist"]
VOLUME_COLS = ["Volume_Surge"]
MACRO_COLS = [
    "Macro_INDIAVIX_Ret1d_L1",
    "Macro_CRUDE_Ret1d_L1",
    "Macro_USDINR_Ret1d_L1",
    "Macro_SP500_Ret1d_L1",
]
ALL_FEATURE_COLS = PRICE_COLS + TECH_COLS + VOLUME_COLS + MACRO_COLS

HMM_REGIME_COL = "HMM_Regime"
TARGET_COL = "Monthly_Alpha"

BENCHMARK_TICKER_STEMS = {"_NSEI", "NSEI", "^NSEI", "NIFTY50", "SP500", "JPM"}
_UNIVERSE_SNAPSHOT_WARNED = False


def _ticker_from_processed_filename(name: str) -> str:
    stem = Path(name).stem
    if stem.endswith("_features"):
        stem = stem[: -len("_features")]
    return stem


def _is_excluded_universe_ticker(stem: str) -> bool:
    return stem.strip().upper() in BENCHMARK_TICKER_STEMS


def build_ticker_universe(processed_dir: str | Path = "data/processed") -> list[str]:
    """Static universe snapshot — matches models/ramt/dataset.build_ticker_universe."""
    global _UNIVERSE_SNAPSHOT_WARNED
    pdir = Path(processed_dir)
    if not pdir.exists():
        return []
    if not _UNIVERSE_SNAPSHOT_WARNED:
        warnings.warn(
            "Tradable universe is a static local snapshot (for this repo, typically the "
            "2026 NIFTY 200 list). Historical index membership is not reconstructed, but "
            "the universe is held fixed across all training dates so the model does not "
            "learn add/delete events such as a 2024 removal during 2015 training.",
            RuntimeWarning,
            stacklevel=2,
        )
        _UNIVERSE_SNAPSHOT_WARNED = True
    tickers: list[str] = []
    for p in sorted(list(pdir.glob("*_features.parquet"))):
        t = _ticker_from_processed_filename(p.name)
        if _is_excluded_universe_ticker(t):
            continue
        tickers.append(t)
    return tickers


def _regime_fallback_from_ret1d(ret1d: pd.Series) -> np.ndarray:
    rv = ret1d.astype(float).rolling(20, min_periods=5).std()
    if rv.isna().any():
        raise ValueError(
            "HMM_Regime is missing and causal fallback would require non-causal backfill. "
            "Regenerate processed features with a valid HMM_Regime column."
        )
    try:
        q = pd.qcut(rv, q=3, labels=[1, 0, 2], duplicates="drop")
        out = pd.to_numeric(q, errors="coerce").fillna(1.0).astype(np.int64).values
        return out
    except Exception:
        med = float(rv.median())
        hi = rv > med * 1.25
        lo = rv < med * 0.75
        return np.where(hi, 2, np.where(lo, 1, 0)).astype(np.int64)


def ensure_hmm_regime_array(df: pd.DataFrame) -> np.ndarray:
    """matches models/ramt/dataset.ensure_hmm_regime_array (no bfill lookahead)."""
    n = len(df)
    if HMM_REGIME_COL in df.columns and df[HMM_REGIME_COL].notna().any():
        s = df[HMM_REGIME_COL].astype(float)
        if s.notna().all():
            return np.clip(s.round().astype(np.int64), 0, 2)
        filled = s.fillna(1.0)
        return np.clip(filled.round().astype(np.int64), 0, 2)
    if "Ret_1d" not in df.columns:
        return np.ones(n, dtype=np.int64)
    return _regime_fallback_from_ret1d(df["Ret_1d"])


class LazyTickerStore:
    """LRU-cached ticker parquet reader — matches models/ramt/dataset.LazyTickerStore."""

    def __init__(self, processed_dir: str | Path = "data/processed", cache_size: int = 6):
        self.processed_dir = Path(processed_dir)
        self.cache_size = int(cache_size)
        self._cache: OrderedDict[str, dict[str, object]] = OrderedDict()

    def _load_ticker_df(self, ticker: str) -> pd.DataFrame:
        path = self.processed_dir / f"{ticker}_features.parquet"
        df = pd.read_parquet(path)
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.sort_values("Date").reset_index(drop=True)
        return df

    def get(self, ticker: str) -> dict[str, object]:
        if ticker in self._cache:
            self._cache.move_to_end(ticker)
            return self._cache[ticker]

        df = self._load_ticker_df(ticker)
        missing = [c for c in ALL_FEATURE_COLS if c not in df.columns]
        if missing:
            raise ValueError(f"{ticker}: missing feature columns: {missing}")
        if TARGET_COL not in df.columns:
            raise ValueError(f"{ticker}: missing {TARGET_COL}")
        if "Daily_Return" not in df.columns:
            if "Ret_1d" not in df.columns:
                raise ValueError(f"{ticker}: need Daily_Return or Ret_1d to build tactical target")
            df = df.copy()
            df["Daily_Return"] = df["Ret_1d"].shift(-1)
        df = df.dropna(subset=[TARGET_COL, "Daily_Return"])

        dates = pd.DatetimeIndex(df["Date"])
        X = df[list(ALL_FEATURE_COLS)].values.astype(np.float32)
        y_m = df[TARGET_COL].values.astype(np.float32)
        y_d = df["Daily_Return"].values.astype(np.float32)
        r = ensure_hmm_regime_array(df.reset_index())

        item = {"dates": dates, "X": X, "y_m": y_m, "y_d": y_d, "r": r}
        self._cache[ticker] = item
        self._cache.move_to_end(ticker)
        while len(self._cache) > self.cache_size:
            self._cache.popitem(last=False)
        return item


class LazyMultiTickerSequenceDataset(Dataset):
    """Sequence dataset backed by per-ticker Parquet features, loaded on demand."""

    def __init__(
        self,
        store: LazyTickerStore,
        sample_keys: list[tuple[str, int]],
        *,
        seq_len: int,
        feature_scaler: RobustScaler | None,
        y_scaler: RobustScaler | None,
        y_winsor_lo: float | None,
        y_winsor_hi: float | None,
        ticker_to_id: dict[str, int],
    ):
        self.store = store
        self.sample_keys = list(sample_keys)
        self.seq_len = int(seq_len)
        self.feature_scaler = feature_scaler
        self.y_scaler = y_scaler
        self.y_winsor_lo = y_winsor_lo
        self.y_winsor_hi = y_winsor_hi
        self.ticker_to_id = dict(ticker_to_id)

    def __len__(self) -> int:
        return len(self.sample_keys)

    def __getitem__(self, idx: int):
        ticker, i = self.sample_keys[idx]
        td = self.store.get(ticker)

        X_raw = td["X"][i - self.seq_len : i]
        if self.feature_scaler is not None:
            X = self.feature_scaler.transform(X_raw.astype(np.float64, copy=False)).astype(np.float32)
        else:
            X = np.asarray(X_raw, dtype=np.float32)

        y_m_raw = float(td["y_m"][i])
        if self.y_winsor_lo is not None and self.y_winsor_hi is not None:
            y_m_raw = float(np.clip(y_m_raw, self.y_winsor_lo, self.y_winsor_hi))
        if self.y_scaler is not None:
            y_m = float(self.y_scaler.transform(np.array([[y_m_raw]], dtype=np.float64))[0, 0])
        else:
            y_m = y_m_raw

        y_d = float(td["y_d"][i])
        r = int(td["r"][i])
        tid = int(self.ticker_to_id.get(ticker, 0))
        d = int(pd.Timestamp(td["dates"][i]).value)

        return (
            torch.from_numpy(X),
            torch.tensor([y_m], dtype=torch.float32),
            torch.tensor([y_d], dtype=torch.float32),
            torch.tensor([r], dtype=torch.long),
            torch.tensor([tid], dtype=torch.long),
            torch.tensor([d], dtype=torch.long),
        )


# -------------------------
# Model architecture (Multimodal + MoE Transformer)
# -------------------------

RET_21D_INPUT_SCALE = 1.65


class FeedForwardEncoder(nn.Module):
    """Projects `input_dim` features per timestep to `embed_dim`."""

    def __init__(self, input_dim: int, embed_dim: int = 32, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class RegimeEncoder(nn.Module):
    """Categorical embedding for HMM regime 0/1/2."""

    def __init__(self, num_regimes: int = 3, embed_dim: int = 32):
        super().__init__()
        self.embedding = nn.Embedding(num_regimes, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 3:
            x = x.squeeze(-1)
        return self.norm(self.embedding(x))


class TickerEncoder(nn.Module):
    """Categorical embedding for a stable cross-stock ticker universe."""

    def __init__(self, num_tickers: int, embed_dim: int = 32):
        super().__init__()
        self.embedding = nn.Embedding(max(1, int(num_tickers)), embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, ticker_ids: torch.Tensor) -> torch.Tensor:
        if ticker_ids.dim() == 2 and ticker_ids.shape[-1] == 1:
            ticker_ids = ticker_ids.squeeze(-1)
        return self.norm(self.embedding(ticker_ids))


class MultimodalEncoder(nn.Module):
    """Encodes lean Parquet features; regime is passed separately and broadcast across time."""

    def __init__(self, *, embed_dim: int = 64, group_dim: int = 32, dropout: float = 0.1, num_tickers: int = 1):
        super().__init__()
        self.embed_dim = int(embed_dim)
        self.group_dim = int(group_dim)

        self.price_encoder = FeedForwardEncoder(len(PRICE_COLS), group_dim, dropout)
        self.tech_encoder = FeedForwardEncoder(len(TECH_COLS), group_dim, dropout)
        self.volume_encoder = FeedForwardEncoder(len(VOLUME_COLS), group_dim, dropout)
        self.macro_encoder = FeedForwardEncoder(len(MACRO_COLS), group_dim, dropout)
        self.regime_encoder = RegimeEncoder(num_regimes=3, embed_dim=group_dim)
        self.ticker_encoder = TickerEncoder(num_tickers=num_tickers, embed_dim=group_dim)

        self._build_indices()

        fusion_input_dim = 6 * group_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
        )

    def _build_indices(self) -> None:
        all_cols = ALL_FEATURE_COLS

        def get_indices(cols: list[str]) -> list[int]:
            return [all_cols.index(c) for c in cols]

        self.price_idx = get_indices(PRICE_COLS)
        self.tech_idx = get_indices(TECH_COLS)
        self.volume_idx = get_indices(VOLUME_COLS)
        self.macro_idx = get_indices(MACRO_COLS)

    def forward(self, x: torch.Tensor, regime: torch.Tensor, *, ticker_id: torch.Tensor | None = None) -> torch.Tensor:
        price_x = x[:, :, self.price_idx].clone()
        price_x[:, :, 2] = price_x[:, :, 2] * RET_21D_INPUT_SCALE

        tech_x = x[:, :, self.tech_idx]
        volume_x = x[:, :, self.volume_idx]
        macro_x = x[:, :, self.macro_idx]

        price_emb = self.price_encoder(price_x)
        tech_emb = self.tech_encoder(tech_x)
        volume_emb = self.volume_encoder(volume_x)
        macro_emb = self.macro_encoder(macro_x)

        regime_seq = regime.long().unsqueeze(1).expand(-1, x.shape[1])
        regime_emb = self.regime_encoder(regime_seq)

        embeddings = [price_emb, tech_emb, volume_emb, macro_emb, regime_emb]
        if ticker_id is not None:
            t_emb = self.ticker_encoder(ticker_id)
            t_emb = t_emb.unsqueeze(1).expand(-1, x.shape[1], -1)
            embeddings.append(t_emb)
        else:
            embeddings.append(torch.zeros_like(price_emb))

        fused = torch.cat(embeddings, dim=-1)
        return self.fusion(fused)


class ExplainableTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """Transformer layer that stores the last attention weights (per head)."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_attn_weights: torch.Tensor | None = None

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal: bool = False):
        attn_output, attn_weights = self.self_attn(
            x,
            x,
            x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
            is_causal=is_causal,
        )
        self.last_attn_weights = attn_weights
        return self.dropout1(attn_output)


class PositionalEncoding(nn.Module):
    """Learnable positional encoding for sequence embeddings."""

    def __init__(self, *, seq_len: int = 30, embed_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.pos_embedding = nn.Embedding(int(seq_len), int(embed_dim))
        self.dropout = nn.Dropout(float(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        if seq_len > self.pos_embedding.num_embeddings:
            raise ValueError(
                f"Input seq_len={seq_len} exceeds positional capacity={self.pos_embedding.num_embeddings}"
            )
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        pos_enc = self.pos_embedding(positions)
        return self.dropout(x + pos_enc)


class ExpertTransformer(nn.Module):
    """A single regime-specialized Transformer expert returning the last-timestep embedding."""

    def __init__(
        self,
        *,
        embed_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        explainable_attn: bool = False,
    ):
        super().__init__()
        layer_cls = ExplainableTransformerEncoderLayer if explainable_attn else nn.TransformerEncoderLayer
        enc_layer = layer_cls(
            d_model=int(embed_dim),
            nhead=int(num_heads),
            dim_feedforward=int(dim_feedforward),
            dropout=float(dropout),
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=int(num_layers))
        self.explainable_attn = bool(explainable_attn)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.transformer(x)
        return out[:, -1, :]

    def get_last_attn_stack(self) -> list[torch.Tensor]:
        if not self.explainable_attn:
            return []
        layers = getattr(self.transformer, "layers", None)
        if layers is None:
            return []
        out: list[torch.Tensor] = []
        for lyr in layers:
            w = getattr(lyr, "last_attn_weights", None)
            if w is not None:
                out.append(w)
        return out


class GatingNetwork(nn.Module):
    """Soft routing weights over experts using context embedding + regime one-hot."""

    def __init__(
        self,
        *,
        embed_dim: int = 64,
        num_regimes: int = 3,
        num_experts: int = 3,
        hidden_dim: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_regimes = int(num_regimes)
        self.num_experts = int(num_experts)

        self.gate = nn.Sequential(
            nn.Linear(int(embed_dim) + int(num_regimes), int(hidden_dim)),
            nn.ReLU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(hidden_dim), int(num_experts)),
        )

    def forward(self, context: torch.Tensor, regime: torch.Tensor) -> torch.Tensor:
        if regime.dim() == 2 and regime.size(-1) == self.num_regimes:
            regime_onehot = regime.float()
        else:
            if regime.dim() > 1:
                regime = regime.squeeze(-1)
            regime_onehot = F.one_hot(regime.long(), num_classes=self.num_regimes).float()
        regime_onehot = regime_onehot.to(context.device)
        logits = self.gate(torch.cat([context, regime_onehot], dim=-1))
        return F.softmax(logits, dim=-1)


class MixtureOfExperts(nn.Module):
    """N expert Transformers blended by a gating network."""

    def __init__(
        self,
        *,
        embed_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 128,
        num_experts: int = 3,
        num_regimes: int = 3,
        dropout: float = 0.1,
        explainable_attn: bool = False,
    ):
        super().__init__()
        self.num_experts = int(num_experts)
        self.explainable_attn = bool(explainable_attn)

        self.experts = nn.ModuleList(
            [
                ExpertTransformer(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    num_layers=num_layers,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                    explainable_attn=explainable_attn,
                )
                for _ in range(int(num_experts))
            ]
        )
        self.gating = GatingNetwork(
            embed_dim=embed_dim,
            num_regimes=num_regimes,
            num_experts=num_experts,
            hidden_dim=32,
            dropout=dropout,
        )

    def forward(
        self,
        x: torch.Tensor,
        regime: torch.Tensor,
        *,
        gating_context: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        context = x[:, -1, :] if gating_context is None else gating_context
        gate_weights = self.gating(context, regime)

        expert_out = [e(x) for e in self.experts]
        expert_stack = torch.stack(expert_out, dim=1)
        fused = (gate_weights.unsqueeze(-1) * expert_stack).sum(dim=1)
        return fused, gate_weights

    def get_last_attention(self) -> list[list[torch.Tensor]]:
        if not self.explainable_attn:
            return []
        return [e.get_last_attn_stack() for e in self.experts]


class RAMTModel(nn.Module):
    """Regime-Adaptive Multimodal Transformer (MoE) with dual heads (monthly + daily)."""

    def __init__(
        self,
        *,
        embed_dim: int = 64,
        group_dim: int = 32,
        num_heads: int = 4,
        num_transformer_layers: int = 2,
        dim_feedforward: int = 128,
        num_experts: int = 3,
        num_regimes: int = 3,
        seq_len: int = 30,
        dropout: float = 0.1,
        explainable_attn: bool = False,
        num_tickers: int = 1,
    ):
        super().__init__()
        self.embed_dim = int(embed_dim)
        self.seq_len = int(seq_len)

        self.encoder = MultimodalEncoder(
            embed_dim=embed_dim,
            group_dim=group_dim,
            dropout=dropout,
            num_tickers=num_tickers,
        )
        self.pos_encoding = PositionalEncoding(seq_len=seq_len, embed_dim=embed_dim, dropout=dropout)
        self.moe = MixtureOfExperts(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_layers=num_transformer_layers,
            dim_feedforward=dim_feedforward,
            num_experts=num_experts,
            num_regimes=num_regimes,
            dropout=dropout,
            explainable_attn=explainable_attn,
        )

        head_hidden = max(8, embed_dim // 2)
        self.monthly_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )
        self.daily_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(
        self, x: torch.Tensor, regime: torch.Tensor, *, ticker_id: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        encoded = self.encoder(x, regime, ticker_id=ticker_id)
        positioned = self.pos_encoding(encoded)
        context, gate_weights = self.moe(positioned, regime)
        pred_m = self.monthly_head(context)
        pred_d = self.daily_head(context)
        return pred_m, pred_d, gate_weights

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_ramt(*, num_tickers: int, config: dict[str, object] | None = None) -> RAMTModel:
    """Factory for `RAMTModel` with sane defaults."""
    # Defaults match models/ramt/moe.py (ExpertTransformer / MixtureOfExperts) + models/ramt/model.py RAMTModel.
    defaults: dict[str, object] = {
        "embed_dim": 64,
        "group_dim": 32,
        "num_heads": 4,
        "num_transformer_layers": 2,
        "dim_feedforward": 128,
        "num_experts": 3,
        "num_regimes": 3,
        "seq_len": 30,
        "dropout": 0.1,
        "explainable_attn": False,
        "num_tickers": int(num_tickers),
    }
    if config:
        defaults.update(config)
    return RAMTModel(**defaults)  # type: ignore[arg-type]


# -------------------------
# Walk-forward training
# -------------------------


@dataclass(frozen=True)
class TrainConfig:
    """Training and walk-forward configuration."""

    seq_len: int = 30
    batch_size: int = 64
    learning_rate: float = 1e-4
    warmup_steps: int = 500
    warmup_lr_start: float = 1e-7
    warmup_lr_end: float = 1e-4
    weight_decay: float = 1e-4
    max_epochs: int = 3
    patience: int = 8
    grad_clip: float = 1.0
    ranking_margin: float = 3.0
    high_vol_sample_weight: float = 2.0
    model_dropout: float = 0.1
    num_heads: int = 4


def _time_decay_weights(db: torch.Tensor) -> torch.Tensor:
    years = pd.to_datetime(db.detach().cpu().numpy()).year.astype(np.int32)
    w = np.ones_like(years, dtype=np.float32)
    w = np.where(years == 2020, 0.5, w)
    w = np.where((years >= 2024) & (years <= 2026), 2.0, w)
    return torch.from_numpy(w).to(db.device)


def _margin_rank_loss(pred: torch.Tensor, y_true: torch.Tensor, *, margin: float) -> torch.Tensor:
    """Tournament mode: MarginRankingLoss; y_true mapped to average relative ranks (ties OK). Same pairwise structure as models/ramt/train_ranking._margin_rank_loss."""
    y_raw = y_true.squeeze(-1)
    r = rankdata(y_raw.detach().cpu().numpy(), method="average")
    denom = max(len(r) - 1, 1)
    y = torch.tensor((r - 1.0) / denom, device=pred.device, dtype=y_raw.dtype)
    p = pred.squeeze(-1)
    n = int(y.shape[0])
    if n < 4:
        return torch.tensor(0.0, device=pred.device)

    k = min(10, n // 2)
    top_idx = torch.topk(y, k=k, largest=True).indices
    bot_idx = torch.topk(y, k=k, largest=False).indices

    top_p = p[top_idx].unsqueeze(1)
    bot_p = p[bot_idx].unsqueeze(0)

    x1 = top_p.expand(-1, bot_p.shape[1]).reshape(-1)
    x2 = bot_p.expand(top_p.shape[0], -1).reshape(-1)
    target = torch.ones_like(x1)
    crit = torch.nn.MarginRankingLoss(margin=float(margin))
    return crit(x1, x2, target)


def _fit_feature_scaler_on_train(
    store: LazyTickerStore,
    train_keys: list[tuple[str, int]],
    *,
    max_fit_samples: int = 200_000,
) -> RobustScaler:
    """Fit RobustScaler on training features only (deterministic subsample for bounded memory)."""
    sc = RobustScaler()
    if not train_keys:
        sc.fit(np.empty((0, len(ALL_FEATURE_COLS)), dtype=np.float32))
        return sc

    step = max(1, int(len(train_keys) / int(max_fit_samples)))
    chosen = train_keys[::step]
    Xb = []
    for t, i in chosen:
        td = store.get(t)
        Xb.append(td["X"][i])
    X = np.asarray(Xb, dtype=np.float32)
    sc.fit(X)
    return sc


def _fit_y_scaler_on_train(store: LazyTickerStore, train_keys: list[tuple[str, int]]) -> RobustScaler:
    ys = []
    for t, i in train_keys:
        td = store.get(t)
        ys.append(float(td["y_m"][i]))
    y_arr = np.asarray(ys, dtype=np.float32).reshape(-1, 1)
    sc = RobustScaler()
    sc.fit(y_arr)
    return sc


def _build_sample_keys_from_store(
    store: LazyTickerStore,
    tickers: list[str],
    *,
    start: str,
    end_inclusive: str,
    seq_len: int,
) -> list[tuple[str, int]]:
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end_inclusive) + pd.Timedelta(days=1)
    keys: list[tuple[str, int]] = []
    for t in tickers:
        td = store.get(t)
        dates: pd.DatetimeIndex = td["dates"]  # type: ignore[assignment]
        mask = (dates >= start_ts) & (dates < end_ts)
        idxs = np.where(mask)[0]
        idxs = idxs[idxs >= int(seq_len)]
        keys.extend([(t, int(i)) for i in idxs])
    return keys


def _train_one_epoch(
    model: RAMTModel,
    loader: DataLoader,
    optimizer: AdamW,
    *,
    cfg: TrainConfig,
    global_step: list[int],
) -> float:
    model.train()
    total = 0.0
    n = 0
    pbar = tqdm(loader, desc="train", leave=False, mininterval=0.5)
    for Xb, yb_m, yb_d, rb, tb, db in pbar:
        if global_step[0] < int(cfg.warmup_steps):
            lr = float(cfg.warmup_lr_start) + (float(cfg.warmup_lr_end) - float(cfg.warmup_lr_start)) * (
                (global_step[0] + 1) / float(cfg.warmup_steps)
            )
            for pg in optimizer.param_groups:
                pg["lr"] = lr

        Xb = Xb.to(DEVICE)
        yb_m = yb_m.to(DEVICE)
        yb_d = yb_d.to(DEVICE)
        rb = rb.squeeze(-1).to(DEVICE)
        tb = tb.squeeze(-1).to(DEVICE)
        db = db.squeeze(-1).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred_m, pred_d, _ = model(Xb, rb, ticker_id=tb)

        # Ranking loss (grouped by date)
        time_w = _time_decay_weights(db).to(dtype=torch.float32)
        rank_losses = []
        rank_w = []
        for d in torch.unique(db):
            m = db == d
            if int(m.sum()) >= 4:
                gw = time_w[m].mean().clamp_min(1e-8)
                rank_losses.append(_margin_rank_loss(pred_m[m], yb_m[m], margin=cfg.ranking_margin) * gw)
                rank_w.append(gw)
        if rank_losses:
            rank_loss = torch.stack(rank_losses).sum() / (torch.stack(rank_w).sum() + 1e-8)
        else:
            rank_loss = torch.tensor(0.0, device=DEVICE)

        # Regime weighting: emphasize high-vol samples
        w = torch.ones_like(rb, dtype=torch.float32, device=DEVICE)
        w = torch.where(rb == 0, torch.tensor(float(cfg.high_vol_sample_weight), device=DEVICE), w)
        w = w * time_w

        # Daily MSE sanity-check
        mse_d = (((pred_d.squeeze(-1) - yb_d.squeeze(-1)) ** 2) * w).sum() / (w.sum() + 1e-8)

        strategic = 0.2 * rank_loss * w.mean()
        loss = 0.7 * strategic + 0.3 * mse_d
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(cfg.grad_clip))
        optimizer.step()
        global_step[0] += 1

        total += float(loss.item())
        n += 1
        pbar.set_postfix(loss=f"{float(loss.item()):.4f}", lr=f"{optimizer.param_groups[0]['lr']:.1e}", refresh=False)

    return total / max(n, 1)


def _eval_loss(model: RAMTModel, loader: DataLoader, *, cfg: TrainConfig) -> float:
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for Xb, yb_m, yb_d, rb, tb, db in loader:
            Xb = Xb.to(DEVICE)
            yb_m = yb_m.to(DEVICE)
            yb_d = yb_d.to(DEVICE)
            rb = rb.squeeze(-1).to(DEVICE)
            tb = tb.squeeze(-1).to(DEVICE)
            db = db.squeeze(-1).to(DEVICE)

            pred_m, pred_d, _ = model(Xb, rb, ticker_id=tb)

            time_w = _time_decay_weights(db).to(dtype=torch.float32)
            rank_losses = []
            rank_w = []
            for d in torch.unique(db):
                m = db == d
                if int(m.sum()) >= 4:
                    gw = time_w[m].mean().clamp_min(1e-8)
                    rank_losses.append(_margin_rank_loss(pred_m[m], yb_m[m], margin=cfg.ranking_margin) * gw)
                    rank_w.append(gw)
            if rank_losses:
                rank_loss = torch.stack(rank_losses).sum() / (torch.stack(rank_w).sum() + 1e-8)
            else:
                rank_loss = torch.tensor(0.0, device=DEVICE)

            w = torch.ones_like(rb, dtype=torch.float32, device=DEVICE)
            w = torch.where(rb == 0, torch.tensor(float(cfg.high_vol_sample_weight), device=DEVICE), w)
            w = w * time_w
            mse_d = (((pred_d.squeeze(-1) - yb_d.squeeze(-1)) ** 2) * w).sum() / (w.sum() + 1e-8)

            strategic = 0.2 * rank_loss * w.mean()
            total += float((0.7 * strategic + 0.3 * mse_d).item())
            n += 1

    return total / max(n, 1)


def save_fold_artifacts(
    *,
    out_dir: str | Path,
    model: RAMTModel,
    scaler: RobustScaler,
    y_scaler: RobustScaler,
    train_start: str,
    train_end: str,
    y_winsor_lo: float,
    y_winsor_hi: float,
    fold_tag: str,
) -> None:
    """Persist fold artifacts into `out_dir` for audit + reuse."""
    out_root = Path(out_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    payload = {
        "model_state_dict": model.state_dict(),
        "config": {"seq_len": int(model.seq_len)},
        "train_start": str(train_start),
        "train_end": str(train_end),
        "y_target_col": TARGET_COL,
        "y_winsor_lo": float(y_winsor_lo),
        "y_winsor_hi": float(y_winsor_hi),
        "fold_tag": str(fold_tag),
    }

    torch.save(payload, out_root / f"ramt_model_state_{_safe_stem(fold_tag)}.pt")
    joblib.dump(scaler, out_root / f"ramt_scaler_{_safe_stem(fold_tag)}.joblib")
    joblib.dump(y_scaler, out_root / f"ramt_y_scaler_{_safe_stem(fold_tag)}.joblib")


def _rebalance_dates_from_nifty_raw(
    nifty_raw_path: str | Path, *, start: str, end: str, step_size: int
) -> pd.DatetimeIndex:
    p = Path(nifty_raw_path)
    df = pd.read_parquet(p, columns=["Date"]) if p.suffix == ".parquet" else pd.read_csv(p, usecols=["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")
    df = df[(df["Date"] >= pd.to_datetime(start)) & (df["Date"] <= pd.to_datetime(end))]
    dates = pd.DatetimeIndex(df["Date"].unique())
    return dates[:: int(step_size)] if len(dates) else pd.DatetimeIndex([])


def combined_walk_forward(
    *,
    processed_dir: str | Path,
    raw_dir: str | Path,
    train_start: str,
    train_end: str,
    end: str,
    training_step: int,
    rebalance_step: int,
    inference_warmup_days: int,
    cfg: TrainConfig,
    artifact_dir: str | Path,
) -> pd.DataFrame:
    """End-to-end walk-forward: retrain every `training_step`, infer every `rebalance_step`."""
    store = LazyTickerStore(processed_dir, cache_size=6)
    tickers = build_ticker_universe(processed_dir)
    if not tickers:
        raise FileNotFoundError(f"No processed feature files found under {Path(processed_dir).resolve()}")

    ticker_to_id = {t: i for i, t in enumerate(tickers)}

    nifty_raw_path = Path(raw_dir) / "_NSEI.parquet"
    if not nifty_raw_path.exists():
        raise FileNotFoundError(f"Missing NIFTY raw parquet at {nifty_raw_path.resolve()}")

    seg_starts = _rebalance_dates_from_nifty_raw(
        nifty_raw_path, start=train_end, end=end, step_size=int(training_step)
    )
    if len(seg_starts) == 0:
        raise RuntimeError("No walk-forward segments: check dates and raw NIFTY calendar.")

    full_cal = pd.read_parquet(nifty_raw_path, columns=["Date"])
    full_cal["Date"] = pd.to_datetime(full_cal["Date"])
    cal = pd.DatetimeIndex(full_cal["Date"].sort_values().unique())

    all_rows: list[dict[str, object]] = []

    for seg_idx, seg_start in enumerate(seg_starts):
        if seg_idx == 0:
            train_end_inclusive = str(pd.Timestamp(train_end).date())
        else:
            sub = cal[cal < pd.Timestamp(seg_start)]
            train_end_inclusive = str((sub[-1] if len(sub) else pd.Timestamp(seg_start)).date())

        fold_label = f"WF_{seg_idx + 1:02d}_train_end_{train_end_inclusive}"
        print(f"\n== {fold_label} ==", flush=True)

        train_keys = _build_sample_keys_from_store(
            store, tickers, start=train_start, end_inclusive=train_end_inclusive, seq_len=cfg.seq_len
        )
        val_start = (pd.Timestamp(train_end_inclusive) - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
        val_keys = _build_sample_keys_from_store(
            store, tickers, start=val_start, end_inclusive=train_end_inclusive, seq_len=cfg.seq_len
        )
        val_set = set(val_keys)
        train_keys_final = [k for k in train_keys if k not in val_set]
        if len(train_keys_final) < 500:
            raise RuntimeError(f"{fold_label}: insufficient training keys ({len(train_keys_final)})")

        scaler = _fit_feature_scaler_on_train(store, train_keys_final)

        # Train-only winsorization bounds (1st/99th percentiles on raw y)
        y_train = np.asarray([float(store.get(t)["y_m"][i]) for t, i in train_keys_final], dtype=np.float32)
        lo_b = float(np.nanpercentile(y_train, 1.0))
        hi_b = float(np.nanpercentile(y_train, 99.0))

        y_scaler = _fit_y_scaler_on_train(store, train_keys_final)

        train_ds = LazyMultiTickerSequenceDataset(
            store,
            sorted(train_keys_final),
            seq_len=cfg.seq_len,
            feature_scaler=scaler,
            y_scaler=y_scaler,
            y_winsor_lo=lo_b,
            y_winsor_hi=hi_b,
            ticker_to_id=ticker_to_id,
        )
        val_ds = LazyMultiTickerSequenceDataset(
            store,
            sorted(val_keys),
            seq_len=cfg.seq_len,
            feature_scaler=scaler,
            y_scaler=y_scaler,
            y_winsor_lo=lo_b,
            y_winsor_hi=hi_b,
            ticker_to_id=ticker_to_id,
        )

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False, num_workers=0)

        model = build_ramt(
            num_tickers=len(tickers),
            config={"seq_len": cfg.seq_len, "num_heads": cfg.num_heads, "dropout": cfg.model_dropout},
        ).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=float(cfg.warmup_lr_end), weight_decay=float(cfg.weight_decay))
        plateau = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-7)
        global_step = [0]

        best = float("inf")
        best_state = None
        patience_ctr = 0

        for epoch in range(int(cfg.max_epochs)):
            tr_loss = _train_one_epoch(model, train_loader, optimizer, cfg=cfg, global_step=global_step)
            v_loss = _eval_loss(model, val_loader, cfg=cfg)
            if global_step[0] >= int(cfg.warmup_steps):
                plateau.step(v_loss)

            lr_now = float(optimizer.param_groups[0]["lr"])
            print(
                f"  epoch {epoch + 1:02d}/{cfg.max_epochs}  train_loss={tr_loss:.6f}  val_loss={v_loss:.6f}  lr={lr_now:.2e}",
                flush=True,
            )

            if v_loss < best:
                best = v_loss
                patience_ctr = 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                patience_ctr += 1
            if patience_ctr >= int(cfg.patience):
                break

        if best_state is not None:
            model.load_state_dict(best_state)

        save_fold_artifacts(
            out_dir=artifact_dir,
            model=model,
            scaler=scaler,
            y_scaler=y_scaler,
            train_start=train_start,
            train_end=train_end_inclusive,
            y_winsor_lo=lo_b,
            y_winsor_hi=hi_b,
            fold_tag=fold_label,
        )

        # Prediction dates for this fold: every `rebalance_step` until next segment.
        seg_end_ts = pd.Timestamp(seg_starts[seg_idx + 1]) if seg_idx + 1 < len(seg_starts) else pd.Timestamp(end) + pd.Timedelta(days=1)
        sub = cal[cal < seg_end_ts]
        pred_end = sub[-1] if len(sub) else (seg_end_ts - pd.Timedelta(days=1))

        pred_dates = _rebalance_dates_from_nifty_raw(
            nifty_raw_path,
            start=str(pd.Timestamp(seg_start).date()),
            end=str(pd.Timestamp(pred_end).date()),
            step_size=int(rebalance_step),
        )

        min_i = int(cfg.seq_len + int(inference_warmup_days))
        rows = []
        for d in tqdm(pred_dates, desc=f"infer[{seg_idx + 1}/{len(seg_starts)}]", leave=False, mininterval=0.5):
            dts = pd.Timestamp(d)
            period = "Train" if dts <= pd.Timestamp(train_end) else "Test"
            for t in tickers:
                td = store.get(t)
                d_arr: pd.DatetimeIndex = td["dates"]  # type: ignore[assignment]
                try:
                    i = int(d_arr.get_loc(dts))
                except KeyError:
                    continue
                if i < min_i:
                    continue
                X_raw = td["X"][i - cfg.seq_len : i]
                Xseq = torch.from_numpy(scaler.transform(X_raw.astype(np.float64)).astype(np.float32)).unsqueeze(0).to(DEVICE)
                r = torch.tensor([int(td["r"][i])], dtype=torch.long).to(DEVICE)
                tid = torch.tensor([int(ticker_to_id.get(t, 0))], dtype=torch.long).to(DEVICE)
                with torch.no_grad():
                    pred_m_sc, _pred_d, _g = model(Xseq, r, ticker_id=tid)
                pred_m = float(y_scaler.inverse_transform([[float(pred_m_sc.detach().cpu().numpy().squeeze())]])[0][0])
                y_raw = float(td["y_m"][i])
                actual_w = float(np.clip(y_raw, lo_b, hi_b))
                rows.append(
                    {
                        "Date": dts,
                        "Ticker": t,
                        "predicted_alpha": float(pred_m),
                        "actual_alpha": float(actual_w),
                        "Period": period,
                        "Fold": fold_label,
                    }
                )

        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df
    df = df.drop_duplicates(subset=["Date", "Ticker"], keep="last")
    return df.sort_values(["Date", "predicted_alpha"], ascending=[True, False])


# -------------------------
# Backtest engine (absolute total return)
# -------------------------

REAL_2026_REBALANCE_FRICTION_RATE = 0.0022


def _load_price_series(raw_dir: str | Path, ticker: str) -> pd.Series:
    path = Path(raw_dir) / f"{ticker}.parquet"
    df = pd.read_parquet(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")
    return df.set_index("Date")["Adj Close"].astype(float)


def build_rebalance_regime_df(
    nifty_features_path: str | Path,
    rebalance_dates: pd.DatetimeIndex,
) -> pd.DataFrame:
    """Regime series aligned to explicit rebalance dates (last available on/before date)."""
    f = pd.read_parquet(nifty_features_path)
    f["Date"] = pd.to_datetime(f["Date"])
    s = f.sort_values("Date").set_index("Date")["HMM_Regime"].astype(float).ffill()
    out = []
    for d in rebalance_dates:
        sel = s.loc[: pd.Timestamp(d)]
        if sel.empty:
            continue
        out.append({"Date": pd.Timestamp(d), "regime": int(sel.iloc[-1])})
    return pd.DataFrame(out)


def run_backtest_absolute_total_return(
    *,
    predictions_df: pd.DataFrame,
    nifty_features_path: str | Path,
    raw_dir: str | Path,
    start: str,
    end: str,
    step_size: int,
    top_n: int = 5,
    capital: float = 100000.0,
    friction_rate: float = REAL_2026_REBALANCE_FRICTION_RATE,
) -> pd.DataFrame:
    """Friction-only rebalanced backtest using realized price returns (no stop-loss, no floors)."""
    preds = predictions_df.copy()
    preds["Date"] = pd.to_datetime(preds["Date"])

    nifty_raw = pd.read_parquet(Path(raw_dir) / "_NSEI.parquet", columns=["Date"])
    nifty_raw["Date"] = pd.to_datetime(nifty_raw["Date"])
    cal = pd.DatetimeIndex(nifty_raw["Date"].sort_values().unique())
    cal = cal[(cal >= pd.to_datetime(start)) & (cal <= pd.to_datetime(end))]
    rebal = cal[:: int(step_size)]
    if len(rebal) < 2:
        return pd.DataFrame()

    regime_df = build_rebalance_regime_df(nifty_features_path, rebal).set_index("Date")["regime"]

    price_cache: dict[str, pd.Series] = {}

    def get_prices(ticker: str) -> pd.Series:
        if ticker not in price_cache:
            price_cache[ticker] = _load_price_series(raw_dir, ticker)
        return price_cache[ticker]

    pv = float(capital)
    results: list[dict[str, object]] = []

    for i in range(len(rebal) - 1):
        d0 = pd.Timestamp(rebal[i])
        d1 = pd.Timestamp(rebal[i + 1])

        regime = int(regime_df.loc[:d0].iloc[-1]) if not regime_df.loc[:d0].empty else 1

        if regime == 2:  # BEAR
            position_size = 0.2
            n_sel = min(5, int(top_n))
        elif regime == 0:  # HIGH_VOL
            position_size = 0.5
            n_sel = 3
        else:  # BULL
            position_size = 1.0
            n_sel = int(top_n)

        day_preds = preds[preds["Date"] == d0].nlargest(n_sel, "predicted_alpha")
        pv_start = float(pv)
        if day_preds.empty:
            results.append(
                {
                    "date": d0,
                    "portfolio_return": 0.0,
                    "portfolio_value_start": pv_start,
                    "portfolio_value": pv,
                    "regime": ["HIGH_VOL", "BULL", "BEAR"][regime],
                    "stocks_held": [],
                    "invested_weight": 0.0,
                    "friction_cost": 0.0,
                }
            )
            continue

        tickers = day_preds["Ticker"].tolist()
        invested = float(position_size)
        w_each = invested / max(1, len(tickers))

        rets = []
        for t in tickers:
            px = get_prices(t)
            window = px.loc[(px.index >= d0) & (px.index < d1)]
            if window.empty:
                rets.append(0.0)
                continue
            entry = float(window.iloc[0])
            exit_px = float(window.iloc[-1])
            rets.append(0.0 if abs(entry) <= 1e-12 else (exit_px - entry) / entry)

        gross = float(np.dot(np.full(len(tickers), w_each, dtype=float), np.asarray(rets, dtype=float)))
        friction_cost = float(pv_start * invested * float(friction_rate))
        port_ret = gross - (friction_cost / pv_start if pv_start > 0 else 0.0)

        pv = pv_start * (1.0 + port_ret)
        results.append(
            {
                "date": d0,
                "portfolio_return": port_ret,
                "portfolio_return_gross": gross,
                "portfolio_value_start": pv_start,
                "portfolio_value": pv,
                "regime": ["HIGH_VOL", "BULL", "BEAR"][regime],
                "stocks_held": tickers,
                "invested_weight": invested,
                "friction_cost": friction_cost,
                "rebalance_friction_rate": float(friction_rate),
            }
        )

    out = pd.DataFrame(results)
    if out.empty:
        return out
    out["cumulative_return"] = out["portfolio_value"] / float(capital) - 1.0
    return out

In [ ]:
# CONFIG — 2020–2026 walk-forward strategy

START_DATE = "2020-01-01"
TRAIN_END = "2023-12-31"
END_DATE_INCLUSIVE = "2026-04-15"

REBALANCE_STEP = 21
TRAINING_STEP = 126

# IO layout (artifacts are written under results/ for easy export)
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RESULTS_DIR = Path("results")

# Training knobs: defaults match models/ramt/moe.py (4 heads, 0.1 dropout). Override to match models/ramt/train_ranking.py (8, 0.2) if you need parity with that script.
TRAIN_CFG = TrainConfig(
    seq_len=30,
    batch_size=64,
    max_epochs=3,
    patience=8,
    num_heads=4,
    model_dropout=0.1,
)

# Backtest parameters
BACKTEST_CAPITAL = 100000.0
BACKTEST_TOP_N = 5
BACKTEST_FRICTION = 0.0022  # 0.22%

print("CONFIG loaded")
print("- START_DATE:", START_DATE)
print("- TRAIN_END:", TRAIN_END)
print("- END_DATE_INCLUSIVE:", END_DATE_INCLUSIVE)
print("- REBALANCE_STEP:", REBALANCE_STEP)
print("- TRAINING_STEP:", TRAINING_STEP)

In [ ]:
# RUN ALL — Download → Features → Walk-Forward Train → Backtest

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("RAMT")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

end_exclusive = (pd.Timestamp(END_DATE_INCLUSIVE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# 1) Download
raw_files = sorted([p for p in RAW_DIR.glob("*.parquet") if p.is_file()])
if len(raw_files) < 10:
    log.info("Downloading universe into %s", RAW_DIR.resolve())
    dl = download_universe_to_parquet(
        raw_dir=RAW_DIR,
        start=START_DATE,
        end_exclusive=end_exclusive,
        rate_limit_s=0.5,
    )
    log.info("Download complete: ok=%s failed=%s", dl["ok"], len(dl["failed"]))
    if dl["failed"]:
        log.warning("Some series failed to download (showing first 10): %s", dl["failed"][:10])
else:
    log.info("Raw parquet already present (%d files). Skipping download.", len(raw_files))

# 2) Process features
log.info("Processing features into %s", PROCESSED_DIR.resolve())
feat = process_features_to_parquet(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    start=START_DATE,
    end_inclusive=END_DATE_INCLUSIVE,
)
log.info("Feature build complete: written=%s skipped=%s", feat["written"], feat["skipped"])

nifty_features_path = PROCESSED_DIR / "_NSEI_features.parquet"
if not nifty_features_path.exists():
    raise FileNotFoundError(f"Missing NIFTY features parquet: {nifty_features_path.resolve()}")

# 3) Walk-forward training + inference
log.info(
    "Walk-forward training: train_start=%s train_end=%s end=%s training_step=%d rebalance_step=%d",
    START_DATE,
    TRAIN_END,
    END_DATE_INCLUSIVE,
    TRAINING_STEP,
    REBALANCE_STEP,
)

preds = combined_walk_forward(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    train_start=START_DATE,
    train_end=TRAIN_END,
    end=END_DATE_INCLUSIVE,
    training_step=TRAINING_STEP,
    rebalance_step=REBALANCE_STEP,
    inference_warmup_days=30,
    cfg=TRAIN_CFG,
    artifact_dir=RESULTS_DIR,
)

preds_path = RESULTS_DIR / "ranking_predictions.csv"
preds.to_csv(preds_path, index=False)
log.info("Saved predictions: %s (rows=%d)", preds_path.resolve(), len(preds))

# Also export a compact per-date ranking view.
rankings = preds.rename(columns={"predicted_alpha": "score"})[["Date", "Ticker", "score", "actual_alpha", "Period", "Fold"]]
rankings_path = RESULTS_DIR / "monthly_rankings.csv"
rankings.to_csv(rankings_path, index=False)
log.info("Saved rankings: %s (rows=%d)", rankings_path.resolve(), len(rankings))

# 4) Backtest (absolute total return, friction-only)
log.info("Running backtest (absolute total return; friction=%.4f)", BACKTEST_FRICTION)

bt = run_backtest_absolute_total_return(
    predictions_df=preds[preds["Period"] == "Test"][["Date", "Ticker", "predicted_alpha", "actual_alpha"]],
    nifty_features_path=nifty_features_path,
    raw_dir=RAW_DIR,
    start=str((pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1)).date()),
    end=END_DATE_INCLUSIVE,
    step_size=REBALANCE_STEP,
    top_n=BACKTEST_TOP_N,
    capital=BACKTEST_CAPITAL,
    friction_rate=BACKTEST_FRICTION,
)

bt_path = RESULTS_DIR / "backtest_results.csv"
bt.to_csv(bt_path, index=False)
log.info("Saved backtest: %s (rows=%d)", bt_path.resolve(), len(bt))

if not bt.empty:
    total = float(bt["portfolio_value"].iloc[-1] / float(BACKTEST_CAPITAL) - 1.0)
    log.info("Backtest summary: final NAV=%.2f total_return=%.2f%%", float(bt["portfolio_value"].iloc[-1]), total * 100.0)

log.info("Pipeline complete.")

In [ ]:
# Artifact export — zip results/ for easy retrieval

zip_path = Path("RAMT_Final_Output.zip").resolve()

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Missing results directory: {RESULTS_DIR.resolve()}")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(RESULTS_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())

print("Created:", str(zip_path))
print("Included files:", len([p for p in RESULTS_DIR.rglob("*") if p.is_file()]))